In [24]:
import joblib

In [25]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "../dataset/processed/feature_engineered_house_prices.csv"
)

print(df.shape)

(2997, 12)


In [26]:
features = [
    "BEDS",
    "BATH",
    "PROPERTYSQFT",
    "TYPE",
    "STATE",
    "LOCALITY"
]

X = df[features]
y = df["PRICE"]

In [27]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [28]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

In [29]:
categorical_features = [
    "TYPE",
    "STATE",
    "LOCALITY"
]

numerical_features = [
    "BEDS",
    "BATH",
    "PROPERTYSQFT"
]

In [30]:
categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                min_frequency=5
            )
        )
    ]
)

numerical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            categorical_transformer,
            categorical_features
        ),
        (
            "numerical",
            numerical_transformer,
            numerical_features
        )
    ]
)

In [31]:
import json

with open(
    "../dataset/processed/best_xgb_params.json",
    "r"
) as f:
    best_params = json.load(f)

print(best_params)

{'model__subsample': 0.9, 'model__n_estimators': 200, 'model__min_child_weight': 5, 'model__max_depth': 8, 'model__learning_rate': 0.03, 'model__colsample_bytree': 0.7}


In [32]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor(
    n_estimators=best_params["model__n_estimators"],
    learning_rate=best_params["model__learning_rate"],
    max_depth=best_params["model__max_depth"],
    min_child_weight=best_params["model__min_child_weight"],
    subsample=best_params["model__subsample"],
    colsample_bytree=best_params["model__colsample_bytree"],
    random_state=42,
    n_jobs=-1,
    objective="reg:squarederror"
)

In [33]:
final_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", xgb_model)
    ]
)

In [34]:
final_model.fit(
    X_train,
    y_train
)

print("Final model trained successfully.")

Final model trained successfully.


In [35]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [36]:
y_pred = final_model.predict(X_test)

In [37]:
mae = mean_absolute_error(
    y_test,
    y_pred
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred
    )
)

r2 = r2_score(
    y_test,
    y_pred
)

In [38]:
print("FINAL MODEL RESULTS")
print("-------------------")
print(f"MAE  : {mae:,.2f}")
print(f"RMSE : {rmse:,.2f}")
print(f"R²   : {r2:.4f}")

FINAL MODEL RESULTS
-------------------
MAE  : 16,928,736.36
RMSE : 27,864,528.77
R²   : 0.8691


In [39]:
joblib.dump(
    final_model,
    "../models/house_price_xgboost.pkl"
)

print("Model saved successfully.")

Model saved successfully.


In [40]:
loaded_model = joblib.load(
    "../models/house_price_xgboost.pkl"
)

print("Model loaded successfully.")

Model loaded successfully.


In [41]:
df.head()

,TYPE,PRICE,BEDS,BATH,PROPERTYSQFT,ADDRESS,STATE,LOCALITY,PRICE_PER_SQFT,SIZE_CATEGORY,BEDS_PER_BATH,TOTAL_BEDS_BATH
0,House for sale,94601238.0,5,5,5938.49,"Moratuwa, Colombo",Colombo,Moratuwa,15930.183936,Very Large,1.00,10
1,House for sale,46044725.0,3,3,1768.45,"Homagama, Colombo",Colombo,Homagama,26036.769487,Medium,1.00,6
2,House for sale,90985626.0,5,4,5063.38,"Boralesgamuwa, Colombo",Colombo,Boralesgamuwa,17969.345773,Very Large,1.25,9
3,House for sale,28967644.0,3,2,2619.82,"Piliyandala, Colombo",Colombo,Piliyandala,11057.112321,Large,1.50,5
4,Villa for sale,98596073.0,5,5,4914.25,"Boralesgamuwa, Colombo",Colombo,Boralesgamuwa,20063.300198,Very Large,1.00,10


In [42]:
new_house = pd.DataFrame({
    "BEDS": [3],
    "BATH": [2],
    "PROPERTYSQFT": [1897],
    "TYPE": ["House for sale"],
    "STATE": ["Colombo"],
    "LOCALITY": ["Moratuwa"]
})

In [43]:
prediction = loaded_model.predict(
    new_house
)

print(
    f"Predicted House Price: "
    f"Rs. {prediction[0]:,.2f}"
)

Predicted House Price: Rs. 33,310,258.00


In [45]:
df.groupby('TYPE')['PRICE'].mean()

TYPE
House for sale    8.659959e+07
Villa for sale    7.975902e+07
Name: PRICE, dtype: float64